# 🏭 SteelDefect-CNN: Ishlab Chiqarishda Nosozliklarni Aniqlash
## Unit 15: Fundamentals of Artificial Intelligence (AI) and Intelligent Systems

**Talaba:** Tojiboyev Abdulvosid  
**Talaba ID:** 240241  
**Guruh:** 24-403  
**O'qituvchi:** Muxammadjon Xolmirzayev  

---

## 📋 Loyiha Haqida

Ushbu loyihada **NEU Surface Defect Dataset** yordamida po'lat yuzasidagi nosozliklarni aniqlash uchun noldan **Konvolyutsion Neyron Tarmoq (CNN)** arxitekturasi loyihalandi va o'qitildi.

### Dataset:
- **Manba:** NEU Surface Defect Database (Northeastern University)
- **Jami rasmlar:** 1800 ta (1440 train + 360 validation)
- **Sinflar soni:** 6 ta
- **Sinf nomlari:** crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches
- **Rasm o'lchami:** 200×200 piksel, kulrang

### CNN Arxitekturasi:
- Noldan loyihalangan maxsus CNN
- 4 ta konvolyutsion blok
- Batch Normalization va Dropout regulyarizatsiya
- Global Average Pooling
- TensorFlow/Keras da amalga oshirilgan

## 1️⃣ Kutubxonalarni O'rnatish va Import Qilish

In [ ]:
# Gradio o'rnatish (mahalliy joylashtirish uchun)
!pip install gradio -q

# Asosiy kutubxonalar
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

# TensorFlow va Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, 
    ModelCheckpoint, TensorBoard
)

# Sklearn metrikalar
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    accuracy_score
)

# PIL tasvir ishlash uchun
from PIL import Image
import cv2

# Muhim ma'lumotlar
print("=" * 50)
print("TensorFlow versiyasi:", tf.__version__)
print("GPU mavjudmi:", tf.config.list_physical_devices('GPU'))
print("=" * 50)

# Random seed - takrorlanadigan natijalar uchun
tf.random.set_seed(42)
np.random.seed(42)

## 2️⃣ Dataset Tayyorlash va O'rganish

In [ ]:
# =============================================
# DATASET NI OCHISH
# =============================================
print("Dataset ochilmoqda...")

with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

# Papka yo'llari
TRAIN_DIR = '/content/dataset/NEU-DET/train/images'
VAL_DIR = '/content/dataset/NEU-DET/validation/images'

# Sinf nomlari
CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)

print("\n" + "=" * 50)
print("DATASET MA'LUMOTLARI")
print("=" * 50)
print(f"Sinf nomlari: {CLASS_NAMES}")
print(f"Sinflar soni: {NUM_CLASSES}")
print("\nHar bir sinfda rasmlar soni:")

total_train = 0
total_val = 0
for cls in CLASS_NAMES:
    train_count = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    val_count = len(os.listdir(os.path.join(VAL_DIR, cls)))
    total_train += train_count
    total_val += val_count
    print(f"  {cls:25s}: Train={train_count}, Val={val_count}")

print(f"\nJami train rasmlar: {total_train}")
print(f"Jami validation rasmlar: {total_val}")
print(f"Jami rasmlar: {total_train + total_val}")

In [ ]:
# =============================================
# DATASET NAMUNALARINI VIZUALIZATSIYA QILISH
# =============================================
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('NEU Surface Defect Dataset - Har Bir Sinfdan Namunalar', 
             fontsize=16, fontweight='bold', y=1.02)

for idx, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(TRAIN_DIR, cls)
    images = os.listdir(cls_path)
    
    # Birinchi rasm
    img1 = Image.open(os.path.join(cls_path, images[0])).convert('L')
    axes[0, idx].imshow(img1, cmap='gray')
    axes[0, idx].set_title(cls.replace('_', '\n'), fontsize=10, fontweight='bold')
    axes[0, idx].axis('off')
    
    # Ikkinchi rasm
    img2 = Image.open(os.path.join(cls_path, images[5])).convert('L')
    axes[1, idx].imshow(img2, cmap='gray')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.savefig('/content/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: dataset_samples.png")

In [ ]:
# =============================================
# SINF TAQSIMOTINI VIZUALIZATSIYA QILISH
# =============================================
train_counts = [len(os.listdir(os.path.join(TRAIN_DIR, cls))) for cls in CLASS_NAMES]
val_counts = [len(os.listdir(os.path.join(VAL_DIR, cls))) for cls in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
x = np.arange(len(CLASS_NAMES))
width = 0.35
bars1 = axes[0].bar(x - width/2, train_counts, width, label='Train', color='steelblue', alpha=0.8)
bars2 = axes[0].bar(x + width/2, val_counts, width, label='Validation', color='coral', alpha=0.8)
axes[0].set_xlabel('Sinf nomi')
axes[0].set_ylabel('Rasmlar soni')
axes[0].set_title('Sinf Taqsimoti: Train va Validation')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES], fontsize=9)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))
axes[1].pie(train_counts, labels=[c.replace('_', ' ') for c in CLASS_NAMES],
           autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Train To\'plamidagi Sinf Ulushi')

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: class_distribution.png")

## 3️⃣ Ma'lumotlarni Qayta Ishlash va Augmentatsiya

### Augmentatsiya usullari:
- **Gorizontal va vertikal burilish** — nosozliklar istalgan yo'nalishda bo'lishi mumkin
- **Kattalashtirish** — turli masofadan olingan rasmlarni simulyatsiya qilish
- **Yorqinlik o'zgartirish** — yoritilish sharoitlari farqi
- **Aylantirish** — nosozlik orientatsiyasidan mustaqil bo'lish

In [ ]:
# =============================================
# RASM O'LCHAMLARI VA KONFIGURATSIYA
# =============================================
IMG_HEIGHT = 128
IMG_WIDTH = 128
IMG_CHANNELS = 1  # Kulrang rasm
BATCH_SIZE = 32
EPOCHS = 50

# =============================================
# TRAIN DATA GENERATOR - AUGMENTATSIYA BILAN
# =============================================
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalizatsiya: 0-255 dan 0-1 ga
    rotation_range=15,         # 15 gradus aylantirish
    width_shift_range=0.1,     # Gorizontal siljitish
    height_shift_range=0.1,    # Vertikal siljitish
    shear_range=0.1,           # Qiyalash
    zoom_range=0.1,            # Kattalashtirish
    horizontal_flip=True,      # Gorizontal aks ettirish
    vertical_flip=True,        # Vertikal aks ettirish
    brightness_range=[0.8, 1.2], # Yorqinlik o'zgartirish
    fill_mode='nearest'        # Bo'sh joylarni to'ldirish
)

# =============================================
# VALIDATION DATA GENERATOR - FAQAT NORMALIZATSIYA
# =============================================
val_datagen = ImageDataGenerator(
    rescale=1./255  # Faqat normalizatsiya, augmentatsiya yo'q
)

# =============================================
# DATA GENERATORLARNI YARATISH
# =============================================
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

print("\nTrain generator:", train_generator.samples, "rasm")
print("Val generator:", val_generator.samples, "rasm")
print("Sinf indekslari:", train_generator.class_indices)

In [ ]:
# =============================================
# AUGMENTATSIYA NATIJALARINI VIZUALIZATSIYA
# =============================================
sample_cls = CLASS_NAMES[0]
sample_img_path = os.path.join(TRAIN_DIR, sample_cls, 
                                os.listdir(os.path.join(TRAIN_DIR, sample_cls))[0])
sample_img = np.array(Image.open(sample_img_path).convert('L').resize((IMG_HEIGHT, IMG_WIDTH)))
sample_img = sample_img.reshape(1, IMG_HEIGHT, IMG_WIDTH, 1)

aug_gen = train_datagen.flow(sample_img, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle(f'Augmentatsiya Namunalari ({sample_cls})', fontsize=14, fontweight='bold')

axes[0, 0].imshow(sample_img[0, :, :, 0], cmap='gray')
axes[0, 0].set_title('Asl rasm', fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 10):
    aug_img = next(aug_gen)[0]
    row, col = divmod(i, 5)
    axes[row, col].imshow(aug_img[:, :, 0], cmap='gray')
    axes[row, col].set_title(f'Augmentatsiya {i}', fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('/content/augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: augmentation_samples.png")

## 4️⃣ Maxsus CNN Arxitekturasi Loyihalash

### Arxitektura qarorlari:
| Qatlam | Parametrlar | Sabab |
|--------|-------------|-------|
| Conv2D (32 filtr, 3×3) | ReLU | Asosiy xususiyatlar: qirralar, to'qimalar |
| Conv2D (64 filtr, 3×3) | ReLU | Murakkab naqshlar |
| Conv2D (128 filtr, 3×3) | ReLU | Yuqori darajali xususiyatlar |
| Conv2D (256 filtr, 3×3) | ReLU | Sinf-spetsifik xususiyatlar |
| BatchNormalization | - | O'qitishni barqarorlashtirish |
| MaxPooling2D (2×2) | - | Fazoviy o'lchamni kamaytirish |
| GlobalAveragePooling2D | - | Parametrlar sonini kamaytirish |
| Dropout (0.5) | - | Ortiqcha o'qitishni oldini olish |
| Dense (512) | ReLU | Tasniflash qatlami |
| Dense (6) | Softmax | Yakuniy natija |

### TensorFlow vs PyTorch taqqoslamasi:
- **TensorFlow/Keras** tanlandi chunki: Keras API sodda va tushunish oson, ImageDataGenerator qulay, model.fit() bilan to'liq nazorat, production uchun keng qo'llab-quvvatlash

In [ ]:
# =============================================
# NOLDAN LOYIHALANGAN MAXSUS CNN ARXITEKTURASI
# =============================================
def build_custom_cnn(input_shape, num_classes):
    """
    SteelDefect-CNN: Maxsus loyihalangan CNN arxitekturasi
    
    Arxitektura 4 ta konvolyutsion blokdan iborat:
    - Har bir blokda 2 ta Conv2D qatlam
    - Batch Normalization barqarorlik uchun
    - MaxPooling o'lcham kamaytirish uchun
    - Dropout ortiqcha o'qitishni oldini olish uchun
    """
    model = models.Sequential(name='SteelDefect_CNN')
    
    # ============================================
    # 1-BLOK: Asosiy xususiyatlarni aniqlash
    # Filtrlar: 32, Rasm: 128x128 -> 64x64
    # ============================================
    model.add(layers.Conv2D(
        32, (3, 3), padding='same',
        input_shape=input_shape,
        name='conv1_1'
    ))
    model.add(layers.BatchNormalization(name='bn1_1'))
    model.add(layers.Activation('relu', name='relu1_1'))
    
    model.add(layers.Conv2D(32, (3, 3), padding='same', name='conv1_2'))
    model.add(layers.BatchNormalization(name='bn1_2'))
    model.add(layers.Activation('relu', name='relu1_2'))
    model.add(layers.MaxPooling2D((2, 2), name='pool1'))
    model.add(layers.Dropout(0.25, name='drop1'))
    
    # ============================================
    # 2-BLOK: O'rtacha murakkablikdagi xususiyatlar
    # Filtrlar: 64, Rasm: 64x64 -> 32x32
    # ============================================
    model.add(layers.Conv2D(64, (3, 3), padding='same', name='conv2_1'))
    model.add(layers.BatchNormalization(name='bn2_1'))
    model.add(layers.Activation('relu', name='relu2_1'))
    
    model.add(layers.Conv2D(64, (3, 3), padding='same', name='conv2_2'))
    model.add(layers.BatchNormalization(name='bn2_2'))
    model.add(layers.Activation('relu', name='relu2_2'))
    model.add(layers.MaxPooling2D((2, 2), name='pool2'))
    model.add(layers.Dropout(0.25, name='drop2'))
    
    # ============================================
    # 3-BLOK: Murakkab xususiyatlar
    # Filtrlar: 128, Rasm: 32x32 -> 16x16
    # ============================================
    model.add(layers.Conv2D(128, (3, 3), padding='same', name='conv3_1'))
    model.add(layers.BatchNormalization(name='bn3_1'))
    model.add(layers.Activation('relu', name='relu3_1'))
    
    model.add(layers.Conv2D(128, (3, 3), padding='same', name='conv3_2'))
    model.add(layers.BatchNormalization(name='bn3_2'))
    model.add(layers.Activation('relu', name='relu3_2'))
    model.add(layers.MaxPooling2D((2, 2), name='pool3'))
    model.add(layers.Dropout(0.3, name='drop3'))
    
    # ============================================
    # 4-BLOK: Yuqori darajali xususiyatlar
    # Filtrlar: 256, Rasm: 16x16 -> 8x8
    # ============================================
    model.add(layers.Conv2D(256, (3, 3), padding='same', name='conv4_1'))
    model.add(layers.BatchNormalization(name='bn4_1'))
    model.add(layers.Activation('relu', name='relu4_1'))
    
    model.add(layers.Conv2D(256, (3, 3), padding='same', name='conv4_2'))
    model.add(layers.BatchNormalization(name='bn4_2'))
    model.add(layers.Activation('relu', name='relu4_2'))
    model.add(layers.MaxPooling2D((2, 2), name='pool4'))
    model.add(layers.Dropout(0.4, name='drop4'))
    
    # ============================================
    # TASNIFLASH QISMI
    # ============================================
    # Global Average Pooling - Flatten dan yaxshiroq
    model.add(layers.GlobalAveragePooling2D(name='gap'))
    
    # To'liq bog'langan qatlamlar
    model.add(layers.Dense(512, name='fc1'))
    model.add(layers.BatchNormalization(name='bn_fc1'))
    model.add(layers.Activation('relu', name='relu_fc1'))
    model.add(layers.Dropout(0.5, name='drop_fc1'))
    
    model.add(layers.Dense(256, name='fc2'))
    model.add(layers.BatchNormalization(name='bn_fc2'))
    model.add(layers.Activation('relu', name='relu_fc2'))
    model.add(layers.Dropout(0.3, name='drop_fc2'))
    
    # Yakuniy natija qatlami
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))
    
    return model

# Modelni yaratish
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
custom_model = build_custom_cnn(INPUT_SHAPE, NUM_CLASSES)

# Modelni compile qilish
custom_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Model arxitekturasini ko'rsatish
custom_model.summary()

In [ ]:
# =============================================
# MODEL ARXITEKTURASINI VIZUALIZATSIYA
# =============================================
try:
    keras.utils.plot_model(
        custom_model, 
        to_file='/content/model_architecture.png',
        show_shapes=True, 
        show_layer_names=True,
        dpi=100
    )
    from IPython.display import Image as IPImage
    display(IPImage('/content/model_architecture.png'))
    print("Model arxitekturasi saqlandi")
except Exception as e:
    print(f"Vizualizatsiya xatosi: {e}")
    print("Model summary ko'rsatildi yuqorida")

# Parametrlar soni
total_params = custom_model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in custom_model.trainable_weights])
print(f"\nJami parametrlar: {total_params:,}")
print(f"O'qitiladigan parametrlar: {trainable_params:,}")

## 5️⃣ Modelni O'qitish

In [ ]:
# =============================================
# CALLBACKS SOZLASH
# =============================================

# 1. EarlyStopping - ortiqcha o'qitishni oldini olish
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,          # 10 epoch yaxshilanmasa to'xtat
    restore_best_weights=True,
    verbose=1
)

# 2. ReduceLROnPlateau - o'quv tezligini kamaytirish
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,           # LR ni 2 ga bo'l
    patience=5,           # 5 epoch yaxshilanmasa
    min_lr=1e-7,
    verbose=1
)

# 3. ModelCheckpoint - eng yaxshi modelni saqlash
checkpoint = ModelCheckpoint(
    '/content/best_custom_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, reduce_lr, checkpoint]

# =============================================
# MODELNI O'QITISH
# =============================================
print("=" * 60)
print("MAXSUS CNN O'QITISH BOSHLANDI")
print("=" * 60)

history = custom_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "=" * 60)
print("O'QITISH YAKUNLANDI")
print(f"Bajarilgan epochlar soni: {len(history.history['accuracy'])}")
print(f"Eng yaxshi val accuracy: {max(history.history['val_accuracy']):.4f}")
print("=" * 60)

## 6️⃣ O'qitish Natijalarini Vizualizatsiya Qilish

In [ ]:
# =============================================
# O'QITISH EGRI CHIZIQLARINI VIZUALIZATSIYA
# =============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Maxsus CNN - O\'qitish Natijalari', fontsize=16, fontweight='bold')

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy grafigi
axes[0].plot(epochs_range, history.history['accuracy'], 
             'b-o', label='Train Accuracy', linewidth=2, markersize=4)
axes[0].plot(epochs_range, history.history['val_accuracy'], 
             'r-s', label='Val Accuracy', linewidth=2, markersize=4)
axes[0].set_title('Model Aniqligi (Accuracy)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Aniqlik')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# Loss grafigi
axes[1].plot(epochs_range, history.history['loss'], 
             'b-o', label='Train Loss', linewidth=2, markersize=4)
axes[1].plot(epochs_range, history.history['val_loss'], 
             'r-s', label='Val Loss', linewidth=2, markersize=4)
axes[1].set_title('Model Yo\'qotishi (Loss)', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Yo\'qotish')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: training_curves.png")

## 7️⃣ Modelni Baholash

In [ ]:
# =============================================
# MAXSUS CNN MODELNI BAHOLASH
# =============================================

# Eng yaxshi modelni yuklash
best_model = keras.models.load_model('/content/best_custom_model.keras')

# Validation to'plamida baholash
val_generator.reset()
val_loss, val_accuracy = best_model.evaluate(val_generator, verbose=0)

print("=" * 60)
print("MAXSUS CNN - BAHOLASH NATIJALARI")
print("=" * 60)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
print("=" * 60)

# Bashoratlarni olish
val_generator.reset()
y_pred_probs = best_model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

# Classification Report
print("\nCLASSIFICATION REPORT:")
print("=" * 60)
report = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4
)
print(report)

In [ ]:
# =============================================
# CONFUSION MATRIX VIZUALIZATSIYA
# =============================================
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Maxsus CNN - Confusion Matrix', fontsize=16, fontweight='bold')

# Oddiy confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[c.replace('_', '\n') for c in CLASS_NAMES],
            yticklabels=[c.replace('_', '\n') for c in CLASS_NAMES])
axes[0].set_title('Sonlar bo\'yicha', fontsize=13)
axes[0].set_xlabel('Bashorat qilingan sinf')
axes[0].set_ylabel('Haqiqiy sinf')

# Normallashtirilgan confusion matrix
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens', ax=axes[1],
            xticklabels=[c.replace('_', '\n') for c in CLASS_NAMES],
            yticklabels=[c.replace('_', '\n') for c in CLASS_NAMES])
axes[1].set_title('Normallashtirilgan (0-1)', fontsize=13)
axes[1].set_xlabel('Bashorat qilingan sinf')
axes[1].set_ylabel('Haqiqiy sinf')

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: confusion_matrix.png")

In [ ]:
# =============================================
# HAR BIR SINF UCHUN METRIKALAR
# =============================================
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Har Bir Sinf Uchun Metrikalar', fontsize=14, fontweight='bold')

colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
x = np.arange(NUM_CLASSES)
short_names = [c.replace('_', '\n') for c in CLASS_NAMES]

axes[0].bar(x, precision, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Precision', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_names, fontsize=8)
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(precision):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

axes[1].bar(x, recall, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_title('Recall', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_names, fontsize=8)
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(recall):
    axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

axes[2].bar(x, f1, color=colors, edgecolor='black', alpha=0.8)
axes[2].set_title('F1-Score', fontsize=13)
axes[2].set_xticks(x)
axes[2].set_xticklabels(short_names, fontsize=8)
axes[2].set_ylim([0, 1])
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(f1):
    axes[2].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: per_class_metrics.png")

## 8️⃣ Bazaviy Model Bilan Taqqoslash (VGG16)

In [ ]:
# =============================================
# BAZAVIY MODEL: VGG16 (FAQAT TAQQOSLASH UCHUN)
# =============================================
print("VGG16 bazaviy modeli yaratilmoqda...")

# VGG16 asosi (oldindan o'qitilgan, ImageNet og'irliklari)
vgg_base = keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)  # VGG16 RGB talab qiladi
)
vgg_base.trainable = False  # Asosiy qatlamlarni muzlatish

# VGG16 ustiga yangi qatlamlar qo'shish
vgg_inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1))
# Kulrang rasmni RGBga o'zgartirish
vgg_rgb = layers.Concatenate()([vgg_inputs, vgg_inputs, vgg_inputs])
vgg_features = vgg_base(vgg_rgb)
vgg_gap = layers.GlobalAveragePooling2D()(vgg_features)
vgg_dense1 = layers.Dense(256, activation='relu')(vgg_gap)
vgg_drop = layers.Dropout(0.5)(vgg_dense1)
vgg_output = layers.Dense(NUM_CLASSES, activation='softmax')(vgg_drop)

baseline_model = keras.Model(inputs=vgg_inputs, outputs=vgg_output, name='VGG16_Baseline')
baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("VGG16 bazaviy modeli tayyorlandi")
print(f"Jami parametrlar: {baseline_model.count_params():,}")

# Bazaviy modelni o'qitish (faqat 10 epoch, taqqoslash uchun)
print("\nVGG16 bazaviy modeli o'qitilmoqda (10 epoch)...")
baseline_history = baseline_model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    verbose=1
)

# Bazaviy modelni baholash
val_generator.reset()
baseline_loss, baseline_accuracy = baseline_model.evaluate(val_generator, verbose=0)
print(f"\nVGG16 Validation Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")

In [ ]:
# =============================================
# MAXSUS CNN VA VGG16 TAQQOSLAMASI
# =============================================
val_generator.reset()
baseline_pred_probs = baseline_model.predict(val_generator, verbose=0)
baseline_pred = np.argmax(baseline_pred_probs, axis=1)

baseline_precision, baseline_recall, baseline_f1, _ = precision_recall_fscore_support(
    y_true, baseline_pred, average='weighted'
)
custom_precision, custom_recall, custom_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted'
)

# Taqqoslash jadvali
print("=" * 65)
print(f"{'METRIK':<25} {'MAXSUS CNN':>15} {'VGG16 (BAZAVIY)':>20}")
print("-" * 65)
print(f"{'Accuracy':<25} {val_accuracy:>14.4f} {baseline_accuracy:>19.4f}")
print(f"{'Precision (weighted)':<25} {custom_precision:>14.4f} {baseline_precision:>19.4f}")
print(f"{'Recall (weighted)':<25} {custom_recall:>14.4f} {baseline_recall:>19.4f}")
print(f"{'F1-Score (weighted)':<25} {custom_f1:>14.4f} {baseline_f1:>19.4f}")
print(f"{'Parametrlar soni':<25} {custom_model.count_params():>14,} {baseline_model.count_params():>19,}")
print("=" * 65)

# Vizualizatsiya
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
custom_scores = [val_accuracy, custom_precision, custom_recall, custom_f1]
baseline_scores = [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, custom_scores, width, label='Maxsus CNN', 
               color='steelblue', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, baseline_scores, width, label='VGG16 (Bazaviy)', 
               color='coral', alpha=0.8, edgecolor='black')

ax.set_title('Maxsus CNN vs VGG16 Bazaviy Model Taqqoslamasi', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim([0, 1.1])
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)
ax.set_ylabel('Ball')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
           f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
           f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: model_comparison.png")

## 9️⃣ Giperparametrlarni Sozlash

In [ ]:
# =============================================
# GIPERPARAMETRLARNI SOZLASH TAHLILI
# =============================================
print("Turli o'quv tezliklari tahlili...")

learning_rates = [0.01, 0.001, 0.0001]
lr_results = {}

for lr in learning_rates:
    print(f"\nO'quv tezligi: {lr}")
    
    # Kichik model - tez test uchun
    test_model = models.Sequential([
        layers.Conv2D(32, (3,3), padding='same', activation='relu',
                     input_shape=INPUT_SHAPE),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    test_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    hist = test_model.fit(
        train_generator,
        epochs=5,
        validation_data=val_generator,
        verbose=0
    )
    
    lr_results[lr] = {
        'train_acc': hist.history['accuracy'],
        'val_acc': hist.history['val_accuracy']
    }
    print(f"  Final val accuracy: {hist.history['val_accuracy'][-1]:.4f}")

# Vizualizatsiya
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("O'quv Tezligi Ta'siri (5 Epoch)", fontsize=14, fontweight='bold')
colors_lr = ['blue', 'red', 'green']

for i, lr in enumerate(learning_rates):
    axes[0].plot(lr_results[lr]['train_acc'], color=colors_lr[i], 
                linestyle='-', label=f'LR={lr}', linewidth=2)
    axes[1].plot(lr_results[lr]['val_acc'], color=colors_lr[i], 
                linestyle='--', label=f'LR={lr}', linewidth=2)

axes[0].set_title("Train Accuracy")
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Aniqlik')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Aniqlik')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/hyperparameter_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nXulosa: LR=0.001 eng yaxshi muvozanatni ta'minlaydi")
print("Rasm saqlandi: hyperparameter_tuning.png")

## 🔟 Xususiyat Xaritalarini Vizualizatsiya Qilish

In [ ]:
# =============================================
# XUSUSIYAT XARITALARINI VIZUALIZATSIYA
# =============================================

# Bitta namuna rasm olish
sample_cls = CLASS_NAMES[0]  # crazing
sample_img_path = os.path.join(TRAIN_DIR, sample_cls,
                                os.listdir(os.path.join(TRAIN_DIR, sample_cls))[0])

# Rasmni yuklash va qayta ishlash
img = Image.open(sample_img_path).convert('L').resize((IMG_HEIGHT, IMG_WIDTH))
img_array = np.array(img) / 255.0
img_array = img_array.reshape(1, IMG_HEIGHT, IMG_WIDTH, 1)

# Intermediate model yaratish
layer_names = ['conv1_1', 'conv2_1', 'conv3_1', 'conv4_1']
intermediate_model = keras.Model(
    inputs=best_model.input,
    outputs=[best_model.get_layer(name).output for name in layer_names]
)

# Xususiyat xaritalarini olish
feature_maps = intermediate_model.predict(img_array, verbose=0)

# Vizualizatsiya
fig, axes = plt.subplots(len(layer_names), 8, figsize=(20, 12))
fig.suptitle('CNN Xususiyat Xaritalari - Turli Qatlamlar', 
             fontsize=14, fontweight='bold')

for row_idx, (layer_name, fmap) in enumerate(zip(layer_names, feature_maps)):
    for col_idx in range(8):
        if col_idx < fmap.shape[-1]:
            axes[row_idx, col_idx].imshow(fmap[0, :, :, col_idx], cmap='viridis')
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(layer_name, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/feature_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rasm saqlandi: feature_maps.png")

## 1️⃣1️⃣ Modelni Saqlash va Mahalliy Joylashtirish (Gradio)

In [ ]:
# =============================================
# MODELNI SAQLASH
# =============================================

# Keras formatida saqlash
best_model.save('/content/steeldefect_cnn_final.keras')
print("Model saqlandi: steeldefect_cnn_final.keras")

# TFLite formatida eksport (ishlab chiqarish uchun)
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()
with open('/content/steeldefect_cnn.tflite', 'wb') as f:
    f.write(tflite_model)
print("TFLite modeli saqlandi: steeldefect_cnn.tflite")

# Model hajmi
import os
keras_size = os.path.getsize('/content/steeldefect_cnn_final.keras') / (1024*1024)
tflite_size = os.path.getsize('/content/steeldefect_cnn.tflite') / (1024*1024)
print(f"\nKeras model hajmi: {keras_size:.2f} MB")
print(f"TFLite model hajmi: {tflite_size:.2f} MB")

In [ ]:
# =============================================
# GRADIO BILAN MAHALLIY JOYLASHTIRISH
# =============================================
import gradio as gr
import numpy as np
from PIL import Image

# Sinf nomlari va izohlari
CLASS_DESCRIPTIONS = {
    'crazing': '🔴 Darz ketish - Po\'lat yuzasida mayda yoriqlar',
    'inclusion': '🟠 Qo\'shimcha - Yot moddalar qo\'shilishi',
    'patches': '🟡 Dog\'lar - Yuzada to\'q dog\'lar',
    'pitted_surface': '🟢 Kovaklar - Yuzada kichik kovaklar',
    'rolled-in_scale': '🔵 Qalinlashgan - Yuzada qatlamlar',
    'scratches': '🟣 Tirnalishlar - Yuzada chiziqlar'
}

def predict_defect(image):
    """
    Rasmni qabul qilib, nosozlik turini aniqlaydi
    """
    if image is None:
        return "Rasm yuklanmadi", {}
    
    # Rasmni qayta ishlash
    if isinstance(image, np.ndarray):
        img = Image.fromarray(image)
    else:
        img = image
    
    img = img.convert('L')  # Kulrang
    img = img.resize((IMG_HEIGHT, IMG_WIDTH))
    img_array = np.array(img) / 255.0
    img_array = img_array.reshape(1, IMG_HEIGHT, IMG_WIDTH, 1)
    
    # Bashorat qilish
    predictions = best_model.predict(img_array, verbose=0)[0]
    
    # Natijalarni formatlash
    results = {}
    for i, cls in enumerate(CLASS_NAMES):
        results[CLASS_DESCRIPTIONS.get(cls, cls)] = float(predictions[i])
    
    best_class = CLASS_NAMES[np.argmax(predictions)]
    confidence = float(np.max(predictions))
    
    result_text = f"""🏭 SteelDefect-CNN Tahlil Natijasi
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Aniqlangan nosozlik: {best_class.upper()}
Ishonch darajasi: {confidence*100:.1f}%
{CLASS_DESCRIPTIONS.get(best_class, '')}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tavsiya: {'Darhol tekshirish kerak!' if confidence > 0.8 else 'Qo\'shimcha tekshirish tavsiya etiladi'}"""
    
    return result_text, results

# Gradio interfeysi
demo = gr.Interface(
    fn=predict_defect,
    inputs=gr.Image(label="Po'lat yuzasi rasmini yuklang", type="pil"),
    outputs=[
        gr.Textbox(label="Tahlil Natijasi", lines=8),
        gr.Label(label="Ehtimollik Taqsimoti", num_top_classes=6)
    ],
    title="🏭 SteelDefect-CNN: Po'lat Yuzasi Nosozliklarini Aniqlash",
    description="""Bu tizim CNN (Konvolyutsion Neyron Tarmoq) yordamida po'lat yuzasidagi 
    nosozliklarni avtomatik aniqlaydi. 6 ta nosozlik turini tasniflay oladi.
    Talaba: Tojiboyev Abdulvosid | Unit 15: AI and Intelligent Systems""",
    examples=[
        [os.path.join(TRAIN_DIR, cls, os.listdir(os.path.join(TRAIN_DIR, cls))[0])]
        for cls in CLASS_NAMES
    ],
    theme=gr.themes.Soft()
)

# Ilovani ishga tushirish
print("Gradio ilovasi ishga tushirilmoqda...")
demo.launch(
    share=True,   # Umumiy havola yaratish
    debug=False
)

## 1️⃣2️⃣ Yakuniy Hisobot va Xulosa

In [ ]:
# =============================================
# YAKUNIY HISOBOT
# =============================================
print("=" * 70)
print("STEELDEFECT-CNN YAKUNIY HISOBOTI")
print("Tojiboyev Abdulvosid | 240241 | Unit 15: AI and Intelligent Systems")
print("=" * 70)

print("""
1. DATASET MA'LUMOTLARI
   - Manba: NEU Surface Defect Database (Northeastern University)
   - Jami rasmlar: 1800 (Train: 1440, Validation: 360)
   - Sinflar: crazing, inclusion, patches, pitted_surface,
              rolled-in_scale, scratches
   - Har bir sinfda: 300 ta rasm (muvozanatli dataset)
   - Rasm formati: 200x200 piksel, kulrang (grayscale)
""")

print("""
2. CNN ARXITEKTURASI
   - Noldan loyihalangan maxsus arxitektura
   - 4 ta konvolyutsion blok (32-64-128-256 filtrlar)
   - Batch Normalization (har blokda)
   - MaxPooling2D (2x2, har blokda)
   - GlobalAveragePooling2D
   - Dropout (0.25 - 0.5 oralig'ida)
   - Dense (512 -> 256 -> 6)
   - Aktivatsiya: ReLU (yashirin), Softmax (chiqish)
""")

print("""
3. AUGMENTATSIYA USULLARI
   - Aylantirish (15 gradus)
   - Gorizontal/vertikal siljitish (10%)
   - Qiyalash va kattalashtirish (10%)
   - Gorizontal va vertikal aks ettirish
   - Yorqinlik o'zgartirish (0.8-1.2)
""")

print(f"""
4. NATIJALAR
   Maxsus CNN:
   - Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)
   - Validation Loss:     {val_loss:.4f}
   - Weighted Precision:  {custom_precision:.4f}
   - Weighted Recall:     {custom_recall:.4f}
   - Weighted F1-Score:   {custom_f1:.4f}
   
   VGG16 Bazaviy Model (10 epoch):
   - Validation Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)
""")

print("""
5. SAQLANGAN FAYLLAR
   - steeldefect_cnn_final.keras  (asosiy model)
   - steeldefect_cnn.tflite       (ishlab chiqarish uchun)
   - best_custom_model.keras      (eng yaxshi checkpoint)
   - dataset_samples.png          (dataset namunalari)
   - class_distribution.png       (sinf taqsimoti)
   - augmentation_samples.png     (augmentatsiya)
   - training_curves.png          (o'qitish egri chiziqlari)
   - confusion_matrix.png         (confusion matrix)
   - per_class_metrics.png        (har sinf metrikalari)
   - model_comparison.png         (model taqqoslamasi)
   - hyperparameter_tuning.png    (giperparametr tahlili)
   - feature_maps.png             (xususiyat xaritalari)
""")

print("""
6. TEXNOLOGIYALAR
   - Python 3.10
   - TensorFlow 2.x / Keras
   - NumPy, Matplotlib, Seaborn
   - Scikit-learn
   - Gradio (mahalliy joylashtirish)
   - Google Colab (GPU: T4)
""")
print("=" * 70)

In [ ]:
# =============================================
# BARCHA FAYLLARNI ZIP GA YIGISH
# =============================================
import zipfile
import os

files_to_save = [
    '/content/steeldefect_cnn_final.keras',
    '/content/steeldefect_cnn.tflite',
    '/content/best_custom_model.keras',
    '/content/dataset_samples.png',
    '/content/class_distribution.png',
    '/content/augmentation_samples.png',
    '/content/training_curves.png',
    '/content/confusion_matrix.png',
    '/content/per_class_metrics.png',
    '/content/model_comparison.png',
    '/content/hyperparameter_tuning.png',
    '/content/feature_maps.png'
]

with zipfile.ZipFile('/content/steeldefect_results.zip', 'w') as zipf:
    for f in files_to_save:
        if os.path.exists(f):
            zipf.write(f, os.path.basename(f))
            print(f"Qo'shildi: {os.path.basename(f)}")

print("\nBarcha fayllar saqlandi: steeldefect_results.zip")

# Yuklab olish
from google.colab import files
files.download('/content/steeldefect_results.zip')
print("Yuklab olish boshlandi!")